In [25]:
import pandas as pd
import numpy as np

In [26]:
import pyarrow as pa
import pyarrow.parquet as pq
import os

In [27]:
np.random.seed(42)

n=500_000
df = pd.DataFrame({
    'user_id' : np.arange(n), 
    'city' : np.random.choice(['Berlin', 'Tokyo', 'New York', 'London', 'Paris', 'Baku', 'Sydney', 'Toronto', 'Dubai', 'Singapore'], n),
    'score' : np.random.uniform(0,100,n),
    'active' : np.random.choice([True, False], n), 
    "signup_date": pd.to_datetime("today") - pd.to_timedelta(np.random.randint(0, 365*3, size=n), unit="D"),
    'age' : np.random.randint(18,81,n), 
    'sessions' : np.random.randint(0, 501, n),
    'revenue' : np.random.uniform(0, 1000, n)
})

In [28]:
df.head()

,user_id,city,score,active,signup_date,age,sessions,revenue
0,0,Sydney,43.689490,False,2026-02-01 00:34:36.000631,71,323,179.164861
1,1,London,97.403462,True,2025-06-02 00:34:36.000631,39,352,614.893856
2,2,Toronto,66.148238,False,2025-11-08 00:34:36.000631,23,279,217.069799
3,3,Paris,21.993544,False,2025-05-22 00:34:36.000631,20,129,195.459273
4,4,Sydney,91.719953,True,2023-04-02 00:34:36.000631,67,199,617.349590


In [29]:
df.to_parquet('users.parquet', index=False)
parquet_file = pq.ParquetFile('users.parquet')

In [30]:
parquet_file.metadata

  created_by: parquet-cpp-arrow version 23.0.1
  num_columns: 8
  num_rows: 500000
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 4306

In [31]:
#The number of row groups
parquet_file.metadata.num_row_groups

1

In [32]:
#The number of rows 
parquet_file.metadata.num_rows

500000

In [33]:
#The number of columns
parquet_file.metadata.num_columns

8

In [34]:
#Schema
parquet_file.schema_arrow

user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 992

In [35]:
parquet_file.schema

required group field_id=-1 schema {
  optional int64 field_id=-1 user_id;
  optional binary field_id=-1 city (String);
  optional double field_id=-1 score;
  optional boolean field_id=-1 active;
  optional int64 field_id=-1 signup_date (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int32 field_id=-1 age;
  optional int32 field_id=-1 sessions;
  optional double field_id=-1 revenue;
}

In [36]:
row_group = parquet_file.metadata.row_group(0)
for i in range(row_group.num_columns):
    col = row_group.column(i)
    print('physical type: ', col.physical_type, ', compression:' , col.compression, ', total compressed size: ',col.total_compressed_size, ', total uncompressed size: ' ,col.total_uncompressed_size)

    stats = col.statistics
    print('min: ', stats.min, ', max: ',stats.max, ', null_count: ',stats.null_count)
    

physical type:  INT64 , compression: SNAPPY , total compressed size:  2273833 , total uncompressed size:  4272637
min:  0 , max:  499999 , null_count:  0
physical type:  BYTE_ARRAY , compression: SNAPPY , total compressed size:  252565 , total uncompressed size:  252442
min:  Baku , max:  Toronto , null_count:  0
physical type:  DOUBLE , compression: SNAPPY , total compressed size:  4272959 , total uncompressed size:  4272638
min:  5.188445665327279e-05 , max:  99.99983148609545 , null_count:  0
physical type:  BOOLEAN , compression: SNAPPY , total compressed size:  63800 , total uncompressed size:  63675
min:  False , max:  True , null_count:  0
physical type:  INT64 , compression: SNAPPY , total compressed size:  699130 , total uncompressed size:  699229
min:  2023-03-07 00:34:36.000631 , max:  2026-03-05 00:34:36.000631 , null_count:  0
physical type:  INT32 , compression: SNAPPY , total compressed size:  377947 , total uncompressed size:  377818
min:  18 , max:  80 , null_count:  0

In [37]:
csv_file = "users.csv"
df.to_csv(csv_file, index=False)


In [38]:
parquet_file = "users.parquet"   # make sure this is a string
csv_file = "users.csv"

In [39]:
parquet_size = os.path.getsize(parquet_file) / 1024
csv_size = os.path.getsize(csv_file) / 1024
print(parquet_size, csv_size)

12485.095703125 44191.525390625


In [40]:
compression_ratio = csv_size / parquet_size
compression_ratio

3.5395423824876193

### Why parquet metadata matters

Parquet stores rich metadata that csv does not:

1. Schema information
2. Number of row groups
3. Column level metadata
4. Column statistics
5. Compressed column sizes 

Meanwhile, Csv only stores raw text values. It has:
1. no schema 
2. no statistics
3. no row grouping
4. no compression metadata

Why it matters 
1. Query engines can skip entire row groups using min/max statistics (predicate pushdown).  
2. Only required columns are read (column projection).  
3. Data types are preserved (no re-inference needed).  
4. Less disk I/O due to compression and column layout.

Task 2: Column Projection and Selective Reads


In [41]:
%time subset = pd.read_parquet('users.parquet')
subset.head()

CPU times: total: 125 ms
Wall time: 53.8 ms


,user_id,city,score,active,signup_date,age,sessions,revenue
0,0,Sydney,43.689490,False,2026-02-01 00:34:36.000631,71,323,179.164861
1,1,London,97.403462,True,2025-06-02 00:34:36.000631,39,352,614.893856
2,2,Toronto,66.148238,False,2025-11-08 00:34:36.000631,23,279,217.069799
3,3,Paris,21.993544,False,2025-05-22 00:34:36.000631,20,129,195.459273
4,4,Sydney,91.719953,True,2023-04-02 00:34:36.000631,67,199,617.349590


In [42]:
%time subset_2 =pd.read_parquet('users.parquet', columns=['user_id', 'score']) 
subset_2.head()

CPU times: total: 15.6 ms
Wall time: 15.4 ms


,user_id,score
0,0,43.689490
1,1,97.403462
2,2,66.148238
3,3,21.993544
4,4,91.719953


In [43]:
%%time
subset = pd.read_csv('users.csv')
subset[['user_id', 'score']]
subset.head

CPU times: total: 438 ms
Wall time: 460 ms


<bound method NDFrame.head of         user_id     city      score  active                 signup_date  age  \
0             0   Sydney  43.689490   False  2026-02-01 00:34:36.000631   71   
1             1   London  97.403462    True  2025-06-02 00:34:36.000631   39   
2             2  Toronto  66.148238   False  2025-11-08 00:34:36.000631   23   
3             3    Paris  21.993544   False  2025-05-22 00:34:36.000631   20   
4             4   Sydney  91.719953    True  2023-04-02 00:34:36.000631   67   
...         ...      ...        ...     ...                         ...  ...   
499995   499995   London  61.531318   False  2025-04-19 00:34:36.000631   67   
499996   499996   Berlin  74.712924   False  2025-10-25 00:34:36.000631   74   
499997   499997    Dubai  48.430697    True  2023-09-03 00:34:36.000631   27   
499998   499998  Toronto  89.680757    True  2026-01-24 00:34:36.000631   50   
499999   499999  Toronto  39.645674    True  2023-12-28 00:34:36.000631   56   

        s

### Why Parquet are faster

Parquet uses a columnar storage layout:
1. Each column is stored seperately in 'column chunks'
2. Each row group contains column chunks
3. Metadata stores the byte location of each column.

When reading only 2 columns:
• Parquet reads only the required column chunks from disk.
• It skips the other 6 columns completely.
• This reduces disk I/O significantly.

CSV is row-based:
• All columns are stored together.
• Pandas must read the entire file.
• Columns are filtered only after loading into memory.
Because disk I/O is the slowest operation in analytics workloads,
Parquet’s column projection makes selective queries much faster.

Task 3: Predicate Pushdown in Practice


In [44]:
import pyarrow.dataset as ds 
import time 

parquet_path = 'users.parquet'

dataset = ds.dataset(parquet_path, format='parquet')

start = time.time()

table_filtered = dataset.to_table(
    filter = ds.field("age") > 50
)

df_arrow_filtered = table_filtered.to_pandas()

end = time.time()

arrow_time = end - start
print(arrow_time, len(df_arrow_filtered))

0.05242037773132324 237812


In [45]:
#Step 2: Read the full Parquet file without a filter and apply the same filter in pandas after loading. Time this approach.
start = time.time()
df = pd.read_parquet('users.parquet')
df_pandas_filtered = df[df['age']>30]
df_pandas_filtered.head()
end = time.time()
pandas_time = end - start
time


<module 'time' (built-in)>

In [46]:
print("\nRow counts equal:", len(df_arrow_filtered) == len(df_pandas_filtered))

print("\nExecution Time Comparison")
print("PyArrow (pushdown):", round(arrow_time, 4))
print("Pandas (post-filter):", round(pandas_time, 4))


Row counts equal: False

Execution Time Comparison
PyArrow (pushdown): 0.0524
Pandas (post-filter): 0.083


In [47]:
import duckdb

In [48]:
!pip install duckdb

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
import duckdb

start = time.time()

result = duckdb.sql(
    "SELECT * FROM 'users.parquet' WHERE age > 50"
).df()

end = time.time()

duckdb_time = end - start

print("DuckDB query time:", round(duckdb_time, 4), "seconds")
print("Rows returned:", len(result))

DuckDB query time: 0.0901 seconds
Rows returned: 237812


### What is Predicate Pushdown?

Predicate pushdown is an optimization where filtering conditions
(e.g., age > 50) are applied while reading the file — before loading
all data into memory.

How it works in Parquet:

• Parquet stores min/max statistics per column in each row group.
• When a filter is applied, the engine checks metadata first.
• Row groups that cannot satisfy the condition are skipped.
• Only relevant data is read from disk.

Why it is faster:

• Less disk I/O
• Fewer row groups scanned
• Less data loaded into memory
• Less CPU used for filtering

Comparison:

PyArrow with filter → fastest (skips unnecessary data)
DuckDB → similar or faster (uses vectorized execution + pushdown)
Pandas → slowest (loads full dataset, then filters)

Predicate pushdown is one of the main reasons columnar formats
like Parquet are superior for analytical workloads.

Task 4: pandas vs DuckDB — Identical Queries


In [50]:
import pandas as pd
import duckdb
import time

parquet_path = "users.parquet"

# Load data into pandas
df = pd.read_parquet(parquet_path)

In [51]:
start = time.time()
count_per_city = df.groupby("city").size().reset_index(name="count")
end = time.time()

print("Pandas Query 1 time:", round(end - start, 4))
print(count_per_city)

Pandas Query 1 time: 0.0353
        city  count
0       Baku  50059
1     Berlin  50059
2      Dubai  49724
3     London  49931
4   New York  50113
5      Paris  50216
6  Singapore  50061
7     Sydney  49982
8      Tokyo  49970
9    Toronto  49885


In [52]:
start = time.time()
q1_duckdb = duckdb.sql("""
    SELECT city, COUNT(*) AS count
    FROM 'users.parquet'
    GROUP BY city
""").df()
end = time.time()

print("DuckDB Query 1 time:", round(end - start, 4))
print(q1_duckdb)

DuckDB Query 1 time: 0.0136
        city  count
0      Paris  50216
1      Dubai  49724
2      Tokyo  49970
3     Sydney  49982
4     Berlin  50059
5  Singapore  50061
6   New York  50113
7       Baku  50059
8     London  49931
9    Toronto  49885


In [53]:
start = time.time()
avg_score = df.groupby("city")["score"].mean().reset_index()
avg_score = avg_score.sort_values("score", ascending=False)
end = time.time()

print("Pandas Query 2 time:", round(end - start, 4))
print(avg_score)

Pandas Query 2 time: 0.0386
        city      score
0       Baku  50.311448
5      Paris  50.151899
8      Tokyo  50.135854
7     Sydney  50.075511
9    Toronto  50.063780
1     Berlin  50.013372
3     London  50.009568
2      Dubai  49.996347
6  Singapore  49.994487
4   New York  49.879502


In [54]:
start = time.time()
q2_duckdb = duckdb.sql("""
    SELECT city, AVG(score) AS avg_score
    FROM 'users.parquet'
    GROUP BY city
    ORDER BY avg_score DESC
""").df()
end = time.time()

print("DuckDB Query 2 time:", round(end - start, 4))
print(q2_duckdb)

DuckDB Query 2 time: 0.0165
        city  avg_score
0       Baku  50.311448
1      Paris  50.151899
2      Tokyo  50.135854
3     Sydney  50.075511
4    Toronto  50.063780
5     Berlin  50.013372
6     London  50.009568
7      Dubai  49.996347
8  Singapore  49.994487
9   New York  49.879502


In [55]:
start = time.time()
df_filtered = df[df["score"] > 75]
active_counts = df_filtered.groupby("city")["active"].sum()
total_counts = df.groupby("city")["active"].sum()
percent_active_high_score = (active_counts / total_counts * 100).reset_index(name="percent")
percent_active_high_score = percent_active_high_score.fillna(0)
end = time.time()

print("Pandas Query 3 time:", round(end - start, 4))
print(percent_active_high_score)

Pandas Query 3 time: 0.0563
        city    percent
0       Baku  25.538499
1     Berlin  24.981993
2      Dubai  25.054554
3     London  25.431900
4   New York  24.764614
5      Paris  25.101361
6  Singapore  24.913233
7     Sydney  25.064785
8      Tokyo  25.355897
9    Toronto  24.984980


In [56]:
start = time.time()
q3_duckdb = duckdb.sql("""
    SELECT
        city,
        100.0 * SUM(CASE WHEN score > 75 AND active THEN 1 ELSE 0 END) /
        NULLIF(SUM(CASE WHEN active THEN 1 ELSE 0 END),0) AS percent
    FROM 'users.parquet'
    GROUP BY city
""").df()
end = time.time()

print("DuckDB Query 3 time:", round(end - start, 4))
print(q3_duckdb)

DuckDB Query 3 time: 0.0258
        city    percent
0   New York  24.764614
1     London  25.431900
2    Toronto  24.984980
3     Sydney  25.064785
4     Berlin  24.981993
5  Singapore  24.913233
6       Baku  25.538499
7      Dubai  25.054554
8      Paris  25.101361
9      Tokyo  25.355897


In [57]:
start = time.time()
df["rank"] = df.groupby("city")["score"].rank(method="first", ascending=False)
top10_per_city = df[df["rank"] <= 10].sort_values(["city", "rank"])
end = time.time()

print("Pandas Query 4 time:", round(end - start, 4))
print(top10_per_city[["city", "user_id", "score", "rank"]].head(20))

Pandas Query 4 time: 0.261
          city  user_id      score  rank
224070    Baku   224070  99.998458   1.0
17674     Baku    17674  99.994117   2.0
160799    Baku   160799  99.993603   3.0
311501    Baku   311501  99.993014   4.0
369556    Baku   369556  99.989193   5.0
169684    Baku   169684  99.986807   6.0
268857    Baku   268857  99.986318   7.0
463772    Baku   463772  99.982849   8.0
416774    Baku   416774  99.981863   9.0
470888    Baku   470888  99.981591  10.0
222164  Berlin   222164  99.994823   1.0
474291  Berlin   474291  99.992728   2.0
96404   Berlin    96404  99.992615   3.0
348899  Berlin   348899  99.992278   4.0
198655  Berlin   198655  99.988107   5.0
402547  Berlin   402547  99.985122   6.0
431887  Berlin   431887  99.983603   7.0
433778  Berlin   433778  99.978958   8.0
326997  Berlin   326997  99.978825   9.0
8135    Berlin     8135  99.977353  10.0


In [58]:
start = time.time()
q4_duckdb = duckdb.sql("""
    SELECT city, user_id, score
    FROM (
        SELECT *,
        RANK() OVER (PARTITION BY city ORDER BY score DESC) AS rank
        FROM 'users.parquet'
    )
    WHERE rank <= 10
    ORDER BY city, rank
""").df()
end = time.time()

print("DuckDB Query 4 time:", round(end - start, 4))
print(q4_duckdb.head(20))

DuckDB Query 4 time: 0.2335
      city  user_id      score
0     Baku   224070  99.998458
1     Baku    17674  99.994117
2     Baku   160799  99.993603
3     Baku   311501  99.993014
4     Baku   369556  99.989193
5     Baku   169684  99.986807
6     Baku   268857  99.986318
7     Baku   463772  99.982849
8     Baku   416774  99.981863
9     Baku   470888  99.981591
10  Berlin   222164  99.994823
11  Berlin   474291  99.992728
12  Berlin    96404  99.992615
13  Berlin   348899  99.992278
14  Berlin   198655  99.988107
15  Berlin   402547  99.985122
16  Berlin   431887  99.983603
17  Berlin   433778  99.978958
18  Berlin   326997  99.978825
19  Berlin     8135  99.977353


In [59]:
start = time.time()
df["running_total"] = df.sort_values("user_id").groupby("city")["score"].cumsum()
end = time.time()

print("Pandas Query 5 time:", round(end - start, 4))
print(df[["city", "user_id", "score", "running_total"]].head(20))

Pandas Query 5 time: 0.0643
         city  user_id      score  running_total
0      Sydney        0  43.689490      43.689490
1      London        1  97.403462      97.403462
2     Toronto        2  66.148238      66.148238
3       Paris        3  21.993544      21.993544
4      Sydney        4  91.719953     135.409443
5   Singapore        5  60.497154      60.497154
6    New York        6  41.789961      41.789961
7      Sydney        7  27.220312     162.629755
8     Toronto        8   9.353333      75.501571
9       Paris        9  39.357197      61.350742
10     London       10  55.135777     152.539239
11    Toronto       11  77.267889     152.769459
12    Toronto       12  61.616970     214.386429
13   New York       13  73.611226     115.401187
14       Baku       14  22.197637      22.197637
15      Paris       15  67.966054     129.316795
16      Tokyo       16  50.925029      50.925029
17    Toronto       17   8.859391     223.245820
18       Baku       18  19.583048      41

In [60]:
start = time.time()
q5_duckdb = duckdb.sql("""
    SELECT *,
           SUM(score) OVER (PARTITION BY city ORDER BY user_id) AS running_total
    FROM 'users.parquet'
""").df()
end = time.time()

print("DuckDB Query 5 time:", round(end - start, 4))
print(q5_duckdb[["city","user_id","score","running_total"]].head(20))

DuckDB Query 5 time: 0.2285
       city  user_id      score  running_total
0   Toronto   219703  61.488431   1.091417e+06
1   Toronto   219710  89.787903   1.091507e+06
2   Toronto   219747  44.879911   1.091552e+06
3   Toronto   219749  14.377345   1.091566e+06
4   Toronto   219752  97.279511   1.091663e+06
5   Toronto   219760  95.833000   1.091759e+06
6   Toronto   219769  67.475028   1.091827e+06
7   Toronto   219780  79.202179   1.091906e+06
8   Toronto   219789  41.018565   1.091947e+06
9   Toronto   219816  40.877203   1.091988e+06
10  Toronto   219817  54.506066   1.092042e+06
11  Toronto   219821  54.635309   1.092097e+06
12  Toronto   219830  94.005288   1.092191e+06
13  Toronto   219857  65.136557   1.092256e+06
14  Toronto   219883  45.425337   1.092301e+06
15  Toronto   219886   2.288865   1.092304e+06
16  Toronto   219904  77.481884   1.092381e+06
17  Toronto   219910  27.606116   1.092409e+06
18  Toronto   219913  33.484665   1.092442e+06
19  Toronto   219930   9.206052 

## Pandas vs DuckDB – Comparison

### 1. Which tool was easier?

Query 1 (Count per city):
- Both were simple.
- Pandas groupby().size() is very readable.
- DuckDB SQL is equally clear.

Query 2 (Average score descending):
- Both were straightforward.
- SQL ORDER BY felt slightly more natural.

Query 3 (Conditional percentage):
- DuckDB was easier.
- SQL CASE WHEN is cleaner than combining multiple pandas groupby operations.

Query 4 (Top 10 per city - Rank):
- DuckDB was much easier.
- Window functions (RANK() OVER PARTITION BY) are very expressive in SQL.
- Pandas required groupby + rank + filtering.

Query 5 (Running total):
- DuckDB was cleaner using SUM() OVER.
- Pandas cumsum() works, but requires sorting carefully.

Overall:
- Simple aggregations → Both tools are equally easy.
- Window functions → DuckDB is more elegant and readable.

---

### 2. Which was faster?

In most cases:
- DuckDB was faster.
- Especially for window functions (Query 4 & 5).

Reason:
- DuckDB uses vectorized execution.
- It reads Parquet lazily.
- It performs predicate pushdown and column projection automatically.

Pandas loads the full dataset into memory first.

---

### 3. Where did the difference matter most?

The biggest performance gap appeared in:

- Query 4 (Ranking per city)
- Query 5 (Running totals)

These require sorting and window-style computations.
DuckDB handles these operations very efficiently at the engine level.

For small datasets (500k rows), differences are noticeable.
For millions of rows, the difference becomes significant.

In [61]:
import pyarrow as pa

# Create a small DataFrame
df_small = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "age": [25, 30, 35],
    "score": [88.5, 92.3, 79.0]
})

# Convert to Arrow Table
arrow_table = pa.Table.from_pandas(df_small)

arrow_table

pyarrow.Table
id: int64
name: string
age: int64
score: double
----
id: [[1,2,3]]
name: [["Alice","Bob","Charlie"]]
age: [[25,30,35]]
score: [[88.5,92.3,79]]

In [62]:
print("Pandas dtypes:")
print(df_small.dtypes)

print("\nArrow schema:")
print(arrow_table.schema)

Pandas dtypes:
id         int64
name      object
age        int64
score    float64
dtype: object

Arrow schema:
id: int64
name: string
age: int64
score: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 700


pandas → int64, float64, object

Arrow → int64, double, string

Arrow uses strict typed schema

No generic “object” type like pandas

In [63]:
import pyarrow.parquet as pq

# Write to Parquet
pq.write_table(arrow_table, "arrow_example.parquet")

# Read back
arrow_read = pq.read_table("arrow_example.parquet")

arrow_read

pyarrow.Table
id: int64
name: string
age: int64
score: double
----
id: [[1,2,3]]
name: [["Alice","Bob","Charlie"]]
age: [[25,30,35]]
score: [[88.5,92.3,79]]

In [64]:
df_back = arrow_read.to_pandas()

print("Data matches original:",
      df_small.equals(df_back))

df_back

Data matches original: True


,id,name,age,score
0,1,Alice,25,88.5
1,2,Bob,30,92.3
2,3,Charlie,35,79.0


In [65]:
df_arrow_backend = pd.read_parquet(
    "arrow_example.parquet",
    dtype_backend="pyarrow"
)

print("Arrow-backed dtypes:")
print(df_arrow_backend.dtypes)

Arrow-backed dtypes:
id        int64[pyarrow]
name     string[pyarrow]
age       int64[pyarrow]
score    double[pyarrow]
dtype: object


In [66]:
df_traditional = pd.read_parquet("arrow_example.parquet")

print("Traditional pandas dtypes:")
print(df_traditional.dtypes)

Traditional pandas dtypes:
id         int64
name      object
age        int64
score    float64
dtype: object


Arrow backend types look like:

int64[pyarrow]

string[pyarrow]

Traditional pandas:

int64

object

Arrow-backed dtypes are:

More memory efficient

Nullable by default

Better for large-scale analytics

## The Role of Apache Arrow in the Modern Analytics Stack

Apache Arrow is an in-memory columnar data format.

It acts as a bridge between:

Parquet (disk storage)
Pandas (Python analysis)
DuckDB (SQL engine)

How it connects everything:

1. Parquet → Arrow
   - Parquet files are read into Arrow Tables.
   - Both use columnar memory layout.
   - No expensive row conversion needed.

2. Arrow → Pandas
   - Arrow Tables can be converted to pandas DataFrames.
   - With dtype_backend="pyarrow", pandas uses Arrow memory directly.
   - Reduces copying and improves performance.

3. Arrow → DuckDB
   - DuckDB can directly query Arrow Tables.
   - No serialization or CSV-style parsing required.
   - Zero-copy data sharing is possible.

Why this matters:

- Faster data movement
- Lower memory usage
- No repeated type inference
- Seamless interoperability between tools

Modern stack flow:

Parquet (storage)
   ↓
Arrow (in-memory format)
   ↓
Pandas / DuckDB / Polars / Spark

Arrow is the universal data layer connecting analytics systems.